# AquaVision - Classifier **augmented** run on Kaggle (Fish4Knowledge, class-balanced)

Same model, input size, epochs and **test split** as the unaugmented baseline (`results/cls_baseline`:
top-1 0.987, macro-F1 0.921, clip-split macro-F1 0.940). What changes is only the TRAIN data and online aug,
following `Augmentation_Brief_Classifier.md`:
* rare/mid classes oversampled to ~300 with species-safe underwater aug (no colour cast), soft cap 2500 on the head
* +20 % augmented copies for every class, cross-domain colour transfer toward Brackish frames
* online: horizontal flip, mild scale, mild S/V jitter (no hue shift - body colour is a species cue)

**Kaggle setup:** GPU T4 x2, Internet On. Attach **`aquavision-scripts`** (with the NEW `f4k_prepare.py`)
and **`aquavision-brackish`**. Run via **Save Version -> Save & Run All (Commit)**.

## 0 - GPU + install

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics
import torch
assert torch.cuda.is_available(), "No CUDA torch - set Accelerator=GPU and do not upgrade torch"
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0))

## 1 - Scripts + config

In [ ]:
import os, shutil, glob
def find_input(name):                                # /kaggle/input/<name> or /kaggle/input/datasets/<user>/<name>
    hits = glob.glob(f"/kaggle/input/{name}") + glob.glob(f"/kaggle/input/*/*/{name}")
    assert hits, f"{name} not attached - Add Input -> {name} (see !ls /kaggle/input)"
    return hits[0]
SCRIPTS = find_input("aquavision-scripts"); RF_BRACKISH = find_input("aquavision-brackish")
print("SCRIPTS =", SCRIPTS, "| contains:", sorted(os.listdir(SCRIPTS)))
NEED = ["f4k_prepare.py", "underwater_aug.py"]
missing = [f for f in NEED if not os.path.exists(f"{SCRIPTS}/{f}")]
assert not missing, f"{missing} not in aquavision-scripts - dataset -> New Version -> add them"
REPO = "/kaggle/working/aquavision"; os.makedirs(REPO, exist_ok=True)
for f in NEED:
    shutil.copy(f"{SCRIPTS}/{f}", f"{REPO}/{f}")
assert "--preview_ids" in open(f"{REPO}/f4k_prepare.py").read(), (
    "aquavision-scripts has the OLD f4k_prepare.py - upload the post-baseline version (dataset -> New Version)")
os.chdir(REPO); print("cwd:", os.getcwd())

F4K  = "/kaggle/working/data/fish4knowledge/fish_image"   # 23 species subfolders
REF  = "/kaggle/working/ref_det"                          # Brackish frames for the colour-transfer stats
WORK = "/kaggle/working/work"
RUNS = "/kaggle/working/runs"
EPOCHS_CLS = 60                                           # = baseline
PREP_ARGS = ("--target 300 --max_per_class 2500 --ceil_mult 10 --max_aug_ratio 12 --extra_aug 0.2 "
             "--preview_ids 8,13,14 --seed 42")

## 2 - Fetch Fish4Knowledge (GitHub)

In [ ]:
os.makedirs("/kaggle/working/data", exist_ok=True)
!git clone --depth 1 https://github.com/Callmewuxin/fish4konwledge /kaggle/working/data/fish4knowledge
print("F4K ->", F4K)

## 3 - Reference frames for cross-domain colour transfer
`f4k_prepare.py --ref_det` reads `<dir>/images/train/*.jpg`. We link 400 Roboflow Brackish frames there
(Brackish-only: DeepFish would need the 7 GB download; Brackish is also the harder, turbid domain).

In [ ]:
import random
from pathlib import Path
src = sorted(p for p in Path(RF_BRACKISH).rglob("*.jpg") if "/train/" in str(p))
dst = Path(REF) / "images" / "train"; shutil.rmtree(REF, ignore_errors=True); dst.mkdir(parents=True)
for i, p in enumerate(random.Random(0).sample(src, min(400, len(src)))):
    os.symlink(p, dst / f"brackish_ref_{i:04d}.jpg")
print(len(list(dst.glob("*.jpg"))), "reference frames ->", dst)

## 4 - Balancing plan (dry run - counts only)
Check: `target=300 | ceiling=2500` (not 381!), rare/mid classes raised toward 300, `real TRAIN crops thrown
away by the cap` is ~6.3k of 19,159 (fish_01 8479 -> 2500; fish_03/04 trimmed slightly), split modes match the baseline.

In [ ]:
!python f4k_prepare.py --f4k "{F4K}" --dry_run {PREP_ARGS}

## 5 - Build the augmented classifier dataset (train only; val/test are the untouched real crops)

In [ ]:
!python f4k_prepare.py --f4k "{F4K}" --out "{WORK}/cls_aug" --ref_det "{REF}" {PREP_ARGS}

# Fair-comparison checks: same TEST set as the baseline, and no synthetic file outside train/
BASE_TEST = {"fish_01": 1211, "fish_02": 268, "fish_03": 359, "fish_04": 405, "fish_05": 253, "fish_06": 19,
             "fish_07": 45, "fish_08": 22, "fish_09": 24, "fish_10": 30, "fish_11": 10, "fish_12": 15,
             "fish_13": 18, "fish_14": 9, "fish_15": 4, "fish_16": 21, "fish_17": 5, "fish_18": 6,
             "fish_19": 3, "fish_20": 2, "fish_21": 2, "fish_22": 4, "fish_23": 2}
D = Path(WORK) / "cls_aug"
test = {c.name: len(list(c.iterdir())) for c in sorted((D / "test").iterdir())}
assert test == BASE_TEST, f"TEST split differs from the baseline: {test}"
synth_leak = [p for s in ("val", "test") for p in (D / s).rglob("aug*__*")]
assert not synth_leak, f"synthetic crops in val/test: {synth_leak[:3]}"
train = {c.name: len(list(c.iterdir())) for c in sorted((D / "train").iterdir())}
print("OK: test split identical to baseline (2737 crops), no synthetic crops in val/test")
print("train per class:", train, "| total", sum(train.values()))

## 6 - Look at the augmented crops (rarest classes + weak mid classes 8, 13, 14) - fish must stay recognisable

In [ ]:
from IPython.display import Image, display
for f in ["rare_class_preview.jpg", "class_balance.png"]:
    p = D / f
    if p.exists(): print(f); display(Image(filename=str(p)))

## 7 - Train (60 epochs, same as baseline) with mild, species-safe online augmentation
Ultralytics' classifier defaults (RandAugment, hsv_s 0.7, erasing 0.4, scale 0.5) would re-introduce the strong
colour changes we removed offline, so every online setting is explicit.

In [ ]:
from ultralytics import YOLO
ONLINE_AUG = dict(fliplr=0.5, flipud=0.0, scale=0.25, hsv_h=0.0, hsv_s=0.2, hsv_v=0.2,
                  erasing=0.1, auto_augment=None, degrees=0.0, translate=0.0)
cls = YOLO("yolo11s-cls.pt")   # ImageNet-pretrained, same start as the baseline
cls.train(data=str(D), epochs=EPOCHS_CLS, imgsz=224, batch=64, device=0, workers=2, project=RUNS,
          name="species_aug", exist_ok=True, save_period=5, **ONLINE_AUG)
CLS_DIR = str(cls.trainer.save_dir); print("run folder:", CLS_DIR)

## 8 - Test: top-1, macro-F1 (+ bootstrap CI), head/mid/tail and clip-split-only macro-F1, vs baseline

In [ ]:
import json, csv, random as _r
import numpy as np
from sklearn.metrics import f1_score, classification_report, confusion_matrix

cm = cls.val(data=str(D), split="test", project=RUNS, name="species_aug_test", exist_ok=True)
names = cls.names; name2id = {v: k for k, v in names.items()}; labels = sorted(names)
imgs = [p for p in (D / "test").rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
y_true, y_pred = [], []
for i in range(0, len(imgs), 256):
    chunk = imgs[i:i + 256]
    for p, r in zip(chunk, cls.predict([str(x) for x in chunk], imgsz=224, verbose=False)):
        y_true.append(name2id[p.parent.name]); y_pred.append(int(r.probs.top1))
y_true, y_pred = np.array(y_true), np.array(y_pred)
mf1 = lambda t, p, lab=labels: f1_score(t, p, labels=lab, average="macro", zero_division=0)

rep = {r["class"]: r for r in csv.DictReader(open(D / "class_report.csv"))}
ids = lambda pred: [name2id[c] for c in rep if pred(rep[c])]
groups = {"head(>=1000 real train)": ids(lambda r: int(r["train_real"]) >= 1000),
          "mid(100-999)": ids(lambda r: 100 <= int(r["train_real"]) < 1000),
          "tail(<100)": ids(lambda r: int(r["train_real"]) < 100),
          "clip_split_only": ids(lambda r: r["split_mode"] == "group")}
rng = _r.Random(0); n = len(y_true); boot = []
for _ in range(1000):
    k = [rng.randrange(n) for _ in range(n)]
    boot.append(mf1(y_true[k], y_pred[k]))
boot.sort()
res = {"top1": round(float(cm.top1), 4), "top5": round(float(cm.top5), 4),
       "macro_f1": round(float(mf1(y_true, y_pred)), 4),
       "macro_f1_ci95": [round(boot[25], 4), round(boot[974], 4)], "n_test": int(n)}
for g, lab in groups.items():
    res[f"macro_f1_{g}"] = round(float(mf1(y_true, y_pred, lab)), 4)
BASE = {"top1": 0.9868, "macro_f1": 0.9208, "macro_f1_clip_split_only": 0.9404,
        "macro_f1_head(>=1000 real train)": 0.992, "macro_f1_mid(100-999)": 0.928, "macro_f1_tail(<100)": 0.879}
print(f"{'metric':34s} {'baseline':>9s} {'augmented':>10s}")
for k, b in BASE.items():
    print(f"{k:34s} {b:9.3f} {res[k]:10.3f}  ({res[k] - b:+.3f})")
print("macro-F1 95% CI:", res["macro_f1_ci95"], " (baseline 0.859-0.956)")
report = classification_report(y_true, y_pred, labels=labels, target_names=[names[i] for i in labels], zero_division=0)
print(report)
json.dump(res, open(f"{CLS_DIR}/test_metrics.json", "w"), indent=1)
Path(f"{CLS_DIR}/classification_report.txt").write_text(report)
json.dump({"labels": [names[i] for i in labels],
           "matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist()},
          open(f"{CLS_DIR}/test_confusion.json", "w"))

## 9 - Files to download
Everything is in `/kaggle/working/runs/species_aug/` (Output tab). `best.pt` ~11 MB.

In [ ]:
for f in ["class_report.csv", "split_counts.json", "data_card.json", "class_balance.png", "rare_class_preview.jpg",
          "class_weights.json"]:
    if (D / f).exists(): shutil.copy(D / f, CLS_DIR)
for f in ["weights/best.pt", "results.csv", "args.yaml", "test_metrics.json", "classification_report.txt",
          "test_confusion.json", "class_report.csv", "rare_class_preview.jpg", "results.png",
          "confusion_matrix_normalized.png"]:
    p = f"{CLS_DIR}/{f}"
    print(f"OK      {p}  ({os.path.getsize(p) / 1e6:.1f} MB)" if os.path.exists(p) else f"MISSING {p}")
shutil.rmtree("/kaggle/working/data", ignore_errors=True)   # raw F4K clone, not needed in Output